# Credit Card Transaction Matching Engine

## Project Purpose

Durnce intern in finance departmentship, credit-card allocation required staff to manually scroll through folio records to locate the transaction corresponding to a card payment.

This notebook develops a transaction-matching engine that searches cleaned synthetic records using available payment details. It ranks possible matches using multiple criteria and provides a confidence score to support human review.

## Matching Criteria

The matching engine will use:

- Last four card digits
- Transaction amount
- Transaction date
- Folio number
- Room number

The last four card digits alone may not uniquely identify a transaction. Combining several criteria helps reduce ambiguous results and improves matching confidence.

## Data Privacy

All records used in this project are synthetic. No actual customer, card or hotel transaction information is included.

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import time
import calendar
import tkinter as tk

from datetime import datetime
from tkinter import ttk, messagebox

# Set the project and data-folder locations
PROJECT_FOLDER = Path.cwd().parent
DATA_FOLDER = PROJECT_FOLDER / "data"

# Load the cleaned and valid transaction records
valid_file = DATA_FOLDER / "valid_transactions.csv"

transactions = pd.read_csv(
    valid_file,
    dtype={
        "transaction_id": "string",
        "card_last4": "string"
    },
    parse_dates=["transaction_date"]
)

print("MATCHING DATASET LOADED")
print("-----------------------")
print(f"Number of searchable transactions: {len(transactions):,}")
print(f"Number of columns: {transactions.shape[1]}")

display(transactions.head())

MATCHING DATASET LOADED
-----------------------
Number of searchable transactions: 1,967
Number of columns: 13


,transaction_id,guest_name,room_no,folio_no,transaction_date,amount_rm,card_last4,payment_method,allocation_status,missing_card,invalid_amount,invalid_date,data_quality_status
0,TXN001199,Guest 0784,498,32386,2025-03-14,727.43,7421,Visa,Pending,False,False,False,Valid
1,TXN000527,Guest 0375,114,30144,2025-03-25,1273.31,1389,Visa,Matched,False,False,False,Valid
2,TXN000394,Guest 0177,148,35550,2025-02-23,814.75,9635,Visa,Matched,False,False,False,Valid
3,TXN001408,Guest 0511,668,36843,2025-03-20,804.24,4383,Mastercard,Matched,False,False,False,Valid
4,TXN000434,Guest 0252,180,33548,2025-01-12,826.32,8010,Visa,Matched,False,False,False,Valid


## Transaction-Matching Logic

The function compares the user’s available payment information against the cleaned transaction records.

Each matching criterion contributes a different weight:

| Matching criterion | Weight |
|---|---:|
| Card last four digits | 40 points |
| Transaction amount | 30 points |
| Transaction date | 20 points |
| Folio number | 5 points |
| Room number | 5 points |

A complete match can receive 100 points. Records are ranked from the highest to the lowest score, but the final selection should still be confirmed by a staff member.

In [14]:
def search_transactions(
    data,
    card_last4=None,
    amount_rm=None,
    transaction_date=None,
    folio_no=None,
    room_no=None,
    maximum_results=10,
    minimum_score=40,
):
    """
    Search and rank possible transaction matches.

    At least one search criterion must be provided.
    """

    # Check that the user entered at least one criterion
    search_inputs = [
        card_last4,
        amount_rm,
        transaction_date,
        folio_no,
        room_no
    ]

    if all(value in [None, ""] for value in search_inputs):
        raise ValueError(
            "Enter at least one search criterion."
        )

    # Create a copy so the original dataset is not changed
    results = data.copy()

    # Start every record with a score of zero
    results["match_score"] = 0

    # Store explanations for each result
    results["match_reasons"] = ""

    # Match card digits: 40 points
    if card_last4 not in [None, ""]:
        card_value = str(card_last4).strip().zfill(4)

        card_match = (
            results["card_last4"].astype("string") == card_value
        )

        results.loc[card_match, "match_score"] += 40
        results.loc[card_match, "match_reasons"] += "Card matched; "

    # Match amount: 30 points
    if amount_rm not in [None, ""]:
        amount_value = float(amount_rm)

        amount_match = np.isclose(
            results["amount_rm"],
            amount_value,
            atol=0.01
        )

        results.loc[amount_match, "match_score"] += 30
        results.loc[amount_match, "match_reasons"] += "Amount matched; "

    # Match date: 20 points
    if transaction_date not in [None, ""]:
        date_value = pd.to_datetime(
            transaction_date,
            errors="coerce"
        )

        if pd.isna(date_value):
            raise ValueError(
                "Enter the date in YYYY-MM-DD format."
            )

        date_match = (
            results["transaction_date"].dt.normalize()
            == date_value.normalize()
        )

        results.loc[date_match, "match_score"] += 20
        results.loc[date_match, "match_reasons"] += "Date matched; "

    # Match folio number: 5 points
    if folio_no not in [None, ""]:
        folio_match = (
            results["folio_no"] == int(folio_no)
        )

        results.loc[folio_match, "match_score"] += 5
        results.loc[folio_match, "match_reasons"] += "Folio matched; "

    # Match room number: 5 points
    if room_no not in [None, ""]:
        room_match = (
            results["room_no"] == int(room_no)
        )

        results.loc[room_match, "match_score"] += 5
        results.loc[room_match, "match_reasons"] += "Room matched; "

    # Keep only records matching at least one criterion
    # Remove weak candidates that do not meet the minimum score
    results = results[
    results["match_score"] >= minimum_score
].copy()
     

    # Classify the strength of each possible match
    results["confidence_level"] = pd.cut(
        results["match_score"],
        bins=[0, 49, 79, 100],
        labels=[
            "Low Confidence",
            "Possible Match",
            "High Confidence"
        ],
        include_lowest=True
    )

    # Rank the best candidates first
    results = results.sort_values(
        by=["match_score", "transaction_date"],
        ascending=[False, False]
    ).head(maximum_results)

    return results[
        [
            "transaction_id",
            "guest_name",
            "room_no",
            "folio_no",
            "transaction_date",
            "amount_rm",
            "card_last4",
            "payment_method",
            "match_score",
            "confidence_level",
            "match_reasons"
        ]
    ]

In [10]:
# Select one existing transaction to simulate a search
test_transaction = transactions.iloc[0]

print("SIMULATED SEARCH INPUT")
print("----------------------")
print("Card:", test_transaction["card_last4"])
print("Amount:", test_transaction["amount_rm"])
print("Date:", test_transaction["transaction_date"].date())

# Search using the simulated payment information
start_time = time.perf_counter()

search_results = search_transactions(
    data=transactions,
    card_last4=test_transaction["card_last4"],
    amount_rm=test_transaction["amount_rm"],
    transaction_date=test_transaction["transaction_date"]
)

search_time = time.perf_counter() - start_time

display(search_results)

print(
    f"Search completed in {search_time:.6f} seconds."
)

SIMULATED SEARCH INPUT
----------------------
Card: 7421
Amount: 727.43
Date: 2025-03-14


,transaction_id,guest_name,room_no,folio_no,transaction_date,amount_rm,card_last4,payment_method,match_score,confidence_level,match_reasons
0,TXN001199,Guest 0784,498,32386,2025-03-14,727.43,7421,Visa,90,High Confidence,Card matched; Amount matched; Date matched;
70,TXN000860,Guest 0357,630,38216,2025-03-14,798.60,7573,American Express,20,Low Confidence,Date matched;
229,TXN000472,Guest 0080,713,31325,2025-03-14,215.20,5046,Mastercard,20,Low Confidence,Date matched;
327,TXN001334,Guest 0679,726,32112,2025-03-14,303.59,8405,Visa,20,Low Confidence,Date matched;
336,TXN001284,Guest 0079,632,30302,2025-03-14,590.68,8923,Visa,20,Low Confidence,Date matched;
382,TXN000428,Guest 0309,438,35902,2025-03-14,278.59,4532,Visa,20,Low Confidence,Date matched;
453,TXN001260,Guest 0329,269,31897,2025-03-14,243.54,6605,American Express,20,Low Confidence,Date matched;
508,TXN000591,Guest 0057,586,30100,2025-03-14,508.89,9669,American Express,20,Low Confidence,Date matched;
583,TXN000227,Guest 0703,415,35953,2025-03-14,249.88,2789,Mastercard,20,Low Confidence,Date matched;
615,TXN001001,Guest 0623,239,39004,2025-03-14,336.25,3561,Mastercard,20,Low Confidence,Date matched;


Search completed in 0.038754 seconds.


## Interactive Transaction Search Form

This interface allows users to enter the available card-payment details without modifying the Python function directly. The system returns ranked candidate transactions together with their scores, confidence levels and matching reasons.

In [21]:
def run_manual_search():
    print("CREDIT CARD TRANSACTION SEARCH")
    print("--------------------------------")
    print("Leave an optional field empty if unavailable.\n")

    card_input = input("Card last 4 digits: ").strip()
    amount_input = input("Amount (RM): ").strip()
    date_input = input("Transaction date (YYYY-MM-DD): ").strip()
    folio_input = input("Folio number (optional): ").strip()
    room_input = input("Room number (optional): ").strip()

    try:
        start_time = time.perf_counter()

        results = search_transactions(
            data=transactions,
            card_last4=card_input or None,
            amount_rm=amount_input or None,
            transaction_date=date_input or None,
            folio_no=folio_input or None,
            room_no=room_input or None,
            minimum_score=40
        )

        elapsed_time = time.perf_counter() - start_time

        print("\nSEARCH RESULTS")
        print("--------------")

        if results.empty:
            print("No sufficiently strong matching transaction was found.")
            print("Check the details or add another search criterion.")
        else:
            print(f"Possible matches found: {len(results)}")
            print(f"Search completed in {elapsed_time:.6f} seconds.\n")
            display(results)

    except ValueError as error:
        print("\nSearch error:", error)

    except Exception as error:
        print("\nUnexpected error:", error)

In [22]:
run_manual_search()

CREDIT CARD TRANSACTION SEARCH
--------------------------------
Leave an optional field empty if unavailable.



Card last 4 digits:  7421
Amount (RM):  727.43
Transaction date (YYYY-MM-DD):  2025-03-14
Folio number (optional):  
Room number (optional):  



SEARCH RESULTS
--------------
Possible matches found: 1
Search completed in 0.046012 seconds.



,transaction_id,guest_name,room_no,folio_no,transaction_date,amount_rm,card_last4,payment_method,match_score,confidence_level,match_reasons
0,TXN001199,Guest 0784,498,32386,2025-03-14,727.43,7421,Visa,90,High Confidence,Card matched; Amount matched; Date matched;


### Search-Test Result

The matching engine was tested using the last four card digits, transaction amount and date from a known synthetic transaction.

The correct transaction was successfully returned with a match score of 90 out of 100 and was classified as a `High Confidence` match. The matched criteria were the card digits, amount and transaction date.

The remaining 10 points were unavailable because the simulated search did not include a folio or room number. This demonstrates that the engine can still identify a strong candidate when only partial payment information is available.

Weak candidates that matched only the transaction date were excluded using a minimum score of 40. This reduced irrelevant results while retaining records with sufficiently useful matching evidence.

The result remains a recommendation for staff verification rather than an automatic financial allocation decision.

## Desktop Search Interface

A Tkinter desktop interface was developed to make the matching engine accessible to users without requiring them to edit Python code.

Users can enter the available transaction information, run a search and review ranked candidate matches with their confidence scores and matching reasons.

In [15]:
def open_calendar(parent, date_variable):

    calendar_window = tk.Toplevel(parent)
    calendar_window.title("Choose Transaction Date")
    calendar_window.resizable(False, False)
    calendar_window.transient(parent)
    calendar_window.grab_set()

    try:
        starting_date = datetime.strptime(
            date_variable.get(),
            "%Y-%m-%d"
        )
    except ValueError:
        starting_date = datetime(2025, 3, 14)

    displayed_year = starting_date.year
    displayed_month = starting_date.month

    calendar_frame = tk.Frame(
        calendar_window,
        padx=12,
        pady=12
    )
    calendar_frame.pack()

    def select_date(day):
        date_variable.set(
            f"{displayed_year}-"
            f"{displayed_month:02d}-"
            f"{day:02d}"
        )
        calendar_window.destroy()

    def change_month(change):
        nonlocal displayed_year, displayed_month

        displayed_month += change

        if displayed_month == 13:
            displayed_month = 1
            displayed_year += 1

        elif displayed_month == 0:
            displayed_month = 12
            displayed_year -= 1

        draw_calendar()

    def draw_calendar():

        for widget in calendar_frame.winfo_children():
            widget.destroy()

        tk.Button(
            calendar_frame,
            text="<",
            command=lambda: change_month(-1)
        ).grid(row=0, column=0)

        tk.Label(
            calendar_frame,
            text=(
                f"{calendar.month_name[displayed_month]} "
                f"{displayed_year}"
            ),
            font=("Arial", 12, "bold"),
            width=20
        ).grid(
            row=0,
            column=1,
            columnspan=5
        )

        tk.Button(
            calendar_frame,
            text=">",
            command=lambda: change_month(1)
        ).grid(row=0, column=6)

        weekdays = [
            "Mon", "Tue", "Wed", "Thu",
            "Fri", "Sat", "Sun"
        ]

        for column, weekday in enumerate(weekdays):
            tk.Label(
                calendar_frame,
                text=weekday,
                font=("Arial", 9, "bold"),
                width=5
            ).grid(row=1, column=column, pady=5)

        month_days = calendar.monthcalendar(
            displayed_year,
            displayed_month
        )

        for row_number, week in enumerate(
            month_days,
            start=2
        ):
            for column_number, day in enumerate(week):

                if day != 0:
                    tk.Button(
                        calendar_frame,
                        text=str(day),
                        width=4,
                        command=lambda chosen_day=day:
                            select_date(chosen_day)
                    ).grid(
                        row=row_number,
                        column=column_number,
                        padx=2,
                        pady=2
                    )

    draw_calendar()

In [17]:
import time
import calendar
import tkinter as tk

from datetime import datetime
from tkinter import ttk, messagebox


def open_calendar(parent, date_variable):
    """Open a calendar and store the selected date."""

    calendar_window = tk.Toplevel(parent)
    calendar_window.title("Choose Transaction Date")
    calendar_window.resizable(False, False)
    calendar_window.transient(parent)
    calendar_window.grab_set()

    try:
        starting_date = datetime.strptime(
            date_variable.get(),
            "%Y-%m-%d"
        )
    except ValueError:
        starting_date = datetime(2025, 3, 14)

    displayed_year = starting_date.year
    displayed_month = starting_date.month

    calendar_frame = tk.Frame(
        calendar_window,
        padx=12,
        pady=12
    )
    calendar_frame.pack()

    def select_date(day):
        selected_date = (
            f"{displayed_year}-"
            f"{displayed_month:02d}-"
            f"{day:02d}"
        )

        date_variable.set(selected_date)
        calendar_window.destroy()

    def change_month(change):
        nonlocal displayed_year, displayed_month

        displayed_month += change

        if displayed_month == 13:
            displayed_month = 1
            displayed_year += 1

        elif displayed_month == 0:
            displayed_month = 12
            displayed_year -= 1

        draw_calendar()

    def draw_calendar():
        for widget in calendar_frame.winfo_children():
            widget.destroy()

        tk.Button(
            calendar_frame,
            text="<",
            width=4,
            command=lambda: change_month(-1)
        ).grid(row=0, column=0)

        tk.Label(
            calendar_frame,
            text=(
                f"{calendar.month_name[displayed_month]} "
                f"{displayed_year}"
            ),
            font=("Arial", 12, "bold"),
            width=22
        ).grid(
            row=0,
            column=1,
            columnspan=5
        )

        tk.Button(
            calendar_frame,
            text=">",
            width=4,
            command=lambda: change_month(1)
        ).grid(row=0, column=6)

        weekdays = [
            "Mon",
            "Tue",
            "Wed",
            "Thu",
            "Fri",
            "Sat",
            "Sun"
        ]

        for column_number, weekday in enumerate(weekdays):
            tk.Label(
                calendar_frame,
                text=weekday,
                font=("Arial", 9, "bold"),
                width=5
            ).grid(
                row=1,
                column=column_number,
                pady=5
            )

        month_days = calendar.monthcalendar(
            displayed_year,
            displayed_month
        )

        for row_number, week in enumerate(
            month_days,
            start=2
        ):
            for column_number, day in enumerate(week):

                if day != 0:
                    tk.Button(
                        calendar_frame,
                        text=str(day),
                        width=4,
                        command=lambda chosen_day=day:
                            select_date(chosen_day)
                    ).grid(
                        row=row_number,
                        column=column_number,
                        padx=2,
                        pady=2
                    )

    draw_calendar()


def open_search_window():
    """Open the credit-card transaction search application."""

    window = tk.Tk()
    window.title("Credit Card Transaction Search")
    window.geometry("1200x650")
    window.configure(bg="#F5F7FA")

    window.attributes("-topmost", True)
    window.after(
        500,
        lambda: window.attributes("-topmost", False)
    )

    # Title
    tk.Label(
        window,
        text="Credit Card Transaction Search",
        font=("Arial", 18, "bold"),
        bg="#F5F7FA",
        fg="#243B53"
    ).pack(pady=(20, 5))

    tk.Label(
        window,
        text=(
            "Enter the available payment information. "
            "More information produces a stronger match."
        ),
        font=("Arial", 10),
        bg="#F5F7FA",
        fg="#52667A"
    ).pack(pady=(0, 15))

    # Input area
    input_frame = tk.Frame(
        window,
        bg="white",
        padx=20,
        pady=15
    )
    input_frame.pack(
        fill="x",
        padx=25,
        pady=5
    )

    field_names = [
        "Card Last 4 Digits",
        "Amount (RM)",
        "Transaction Date",
        "Folio Number",
        "Room Number"
    ]

    for column_number, field_name in enumerate(field_names):
        tk.Label(
            input_frame,
            text=field_name,
            bg="white",
            font=("Arial", 9, "bold")
        ).grid(
            row=0,
            column=column_number,
            padx=8,
            pady=5
        )

    card_entry = tk.Entry(
        input_frame,
        width=18
    )

    amount_entry = tk.Entry(
        input_frame,
        width=18
    )

    date_variable = tk.StringVar(
        value="2025-03-14"
    )

    date_frame = tk.Frame(
        input_frame,
        bg="white"
    )

    date_entry = tk.Entry(
        date_frame,
        width=12,
        textvariable=date_variable,
        state="readonly"
    )
    date_entry.pack(side="left")

    calendar_button = tk.Button(
        date_frame,
        text="Calendar",
        command=lambda: open_calendar(
            window,
            date_variable
        )
    )
    calendar_button.pack(
        side="left",
        padx=(4, 0)
    )

    folio_entry = tk.Entry(
        input_frame,
        width=18
    )

    room_entry = tk.Entry(
        input_frame,
        width=18
    )

    card_entry.grid(
        row=1,
        column=0,
        padx=8,
        pady=5
    )

    amount_entry.grid(
        row=1,
        column=1,
        padx=8,
        pady=5
    )

    date_frame.grid(
        row=1,
        column=2,
        padx=8,
        pady=5
    )

    folio_entry.grid(
        row=1,
        column=3,
        padx=8,
        pady=5
    )

    room_entry.grid(
        row=1,
        column=4,
        padx=8,
        pady=5
    )

    # Results table
    result_frame = tk.Frame(
        window,
        bg="#F5F7FA"
    )
    result_frame.pack(
        fill="both",
        expand=True,
        padx=25,
        pady=15
    )

    columns = [
        "transaction_id",
        "guest_name",
        "room_no",
        "folio_no",
        "transaction_date",
        "amount_rm",
        "card_last4",
        "match_score",
        "confidence_level",
        "match_reasons"
    ]

    headings = {
        "transaction_id": "Transaction ID",
        "guest_name": "Guest",
        "room_no": "Room",
        "folio_no": "Folio",
        "transaction_date": "Date",
        "amount_rm": "Amount (RM)",
        "card_last4": "Card Last 4",
        "match_score": "Score",
        "confidence_level": "Confidence",
        "match_reasons": "Matching Reasons"
    }

    widths = {
        "transaction_id": 110,
        "guest_name": 110,
        "room_no": 70,
        "folio_no": 80,
        "transaction_date": 100,
        "amount_rm": 100,
        "card_last4": 90,
        "match_score": 65,
        "confidence_level": 120,
        "match_reasons": 300
    }

    result_table = ttk.Treeview(
        result_frame,
        columns=columns,
        show="headings"
    )

    for column in columns:
        result_table.heading(
            column,
            text=headings[column]
        )
        result_table.column(
            column,
            width=widths[column],
            anchor="center"
        )

    vertical_scrollbar = ttk.Scrollbar(
        result_frame,
        orient="vertical",
        command=result_table.yview
    )

    horizontal_scrollbar = ttk.Scrollbar(
        result_frame,
        orient="horizontal",
        command=result_table.xview
    )

    result_table.configure(
        yscrollcommand=vertical_scrollbar.set,
        xscrollcommand=horizontal_scrollbar.set
    )

    result_table.grid(
        row=0,
        column=0,
        sticky="nsew"
    )

    vertical_scrollbar.grid(
        row=0,
        column=1,
        sticky="ns"
    )

    horizontal_scrollbar.grid(
        row=1,
        column=0,
        sticky="ew"
    )

    result_frame.rowconfigure(0, weight=1)
    result_frame.columnconfigure(0, weight=1)

    status_label = tk.Label(
        window,
        text="Enter search information to begin.",
        font=("Arial", 10),
        bg="#F5F7FA",
        fg="#52667A"
    )
    status_label.pack(pady=(0, 15))

    def perform_gui_search():
        for item in result_table.get_children():
            result_table.delete(item)

        try:
            start_time = time.perf_counter()

            results = search_transactions(
                data=transactions,
                card_last4=card_entry.get().strip() or None,
                amount_rm=amount_entry.get().strip() or None,
                transaction_date=date_variable.get().strip() or None,
                folio_no=folio_entry.get().strip() or None,
                room_no=room_entry.get().strip() or None,
                minimum_score=40
            )

            elapsed_time = time.perf_counter() - start_time

            if results.empty:
                status_label.config(
                    text=(
                        "No sufficiently strong match was found. "
                        f"Search time: {elapsed_time:.6f} seconds."
                    ),
                    fg="#C0392B"
                )
                return

            for _, row in results.iterrows():
                displayed_date = pd.to_datetime(
                    row["transaction_date"]
                ).strftime("%Y-%m-%d")

                result_table.insert(
                    "",
                    "end",
                    values=(
                        row["transaction_id"],
                        row["guest_name"],
                        row["room_no"],
                        row["folio_no"],
                        displayed_date,
                        f"{row['amount_rm']:.2f}",
                        row["card_last4"],
                        row["match_score"],
                        str(row["confidence_level"]),
                        row["match_reasons"]
                    )
                )

            status_label.config(
                text=(
                    f"{len(results)} possible match(es) found in "
                    f"{elapsed_time:.6f} seconds."
                ),
                fg="#1B7F5A"
            )

        except ValueError as error:
            messagebox.showerror(
                "Invalid Search",
                str(error)
            )

        except Exception as error:
            messagebox.showerror(
                "Unexpected Error",
                str(error)
            )

    def clear_search():
        card_entry.delete(0, tk.END)
        amount_entry.delete(0, tk.END)
        date_variable.set("")
        folio_entry.delete(0, tk.END)
        room_entry.delete(0, tk.END)

        for item in result_table.get_children():
            result_table.delete(item)

        status_label.config(
            text="Enter search information to begin.",
            fg="#52667A"
        )

        card_entry.focus()

    # Buttons
    button_frame = tk.Frame(
        input_frame,
        bg="white"
    )
    button_frame.grid(
        row=2,
        column=0,
        columnspan=5,
        pady=(15, 0)
    )

    tk.Button(
        button_frame,
        text="Search Transactions",
        command=perform_gui_search,
        bg="#2A9D8F",
        fg="white",
        font=("Arial", 10, "bold"),
        padx=18,
        pady=7
    ).pack(side="left", padx=5)

    tk.Button(
        button_frame,
        text="Clear",
        command=clear_search,
        bg="#DCE3EA",
        fg="#243B53",
        font=("Arial", 10),
        padx=18,
        pady=7
    ).pack(side="left", padx=5)

    card_entry.focus()
    window.mainloop()

In [19]:
open_search_window()

## Matching-Engine Testing

The matching engine was tested using controlled scenarios to verify that it handles complete, partial, ambiguous and incorrect information appropriately.

The test scenarios include:

1. Exact match using all five criteria.
2. Partial match using card digits and amount.
3. Ambiguous match using card digits only.
4. No matching transaction.
5. Invalid input.

In [20]:
# Select a known transaction for controlled testing
known_transaction = transactions.iloc[0]

exact_match_result = search_transactions(
    data=transactions,
    card_last4=known_transaction["card_last4"],
    amount_rm=known_transaction["amount_rm"],
    transaction_date=known_transaction["transaction_date"],
    folio_no=known_transaction["folio_no"],
    room_no=known_transaction["room_no"],
    minimum_score=40
)

print("TEST 1: EXACT MATCH")
print("-------------------")
print("Expected transaction:", known_transaction["transaction_id"])

display(exact_match_result)

TEST 1: EXACT MATCH
-------------------
Expected transaction: TXN001199


,transaction_id,guest_name,room_no,folio_no,transaction_date,amount_rm,card_last4,payment_method,match_score,confidence_level,match_reasons
0,TXN001199,Guest 0784,498,32386,2025-03-14,727.43,7421,Visa,100,High Confidence,Card matched; Amount matched; Date matched; Fo...


In [21]:
partial_match_result = search_transactions(
    data=transactions,
    card_last4=known_transaction["card_last4"],
    amount_rm=known_transaction["amount_rm"],
    minimum_score=40
)

print("TEST 2: PARTIAL MATCH")
print("---------------------")
print("Expected transaction:", known_transaction["transaction_id"])

display(partial_match_result)

TEST 2: PARTIAL MATCH
---------------------
Expected transaction: TXN001199


,transaction_id,guest_name,room_no,folio_no,transaction_date,amount_rm,card_last4,payment_method,match_score,confidence_level,match_reasons
0,TXN001199,Guest 0784,498,32386,2025-03-14,727.43,7421,Visa,70,Possible Match,Card matched; Amount matched;


In [22]:
# Find card digits appearing in multiple transactions
card_frequencies = (
    transactions["card_last4"]
    .value_counts()
)

repeated_cards = card_frequencies[
    card_frequencies > 1
]

if repeated_cards.empty:
    print("No repeated card digits were found.")

else:
    ambiguous_card = repeated_cards.index[0]

    ambiguous_result = search_transactions(
        data=transactions,
        card_last4=ambiguous_card,
        minimum_score=40
    )

    print("TEST 3: AMBIGUOUS MATCH")
    print("-----------------------")
    print("Card searched:", ambiguous_card)
    print(
        "Transactions containing this card:",
        repeated_cards.iloc[0]
    )

    display(ambiguous_result)

TEST 3: AMBIGUOUS MATCH
-----------------------
Card searched: 0516
Transactions containing this card: 3


,transaction_id,guest_name,room_no,folio_no,transaction_date,amount_rm,card_last4,payment_method,match_score,confidence_level,match_reasons
1700,TXN000642,Guest 0398,123,30124,2025-03-24,284.15,0516,Visa,40,Low Confidence,Card matched;
44,TXN000299,Guest 0178,126,32961,2025-01-26,448.03,0516,Visa,40,Low Confidence,Card matched;
1536,TXN000993,Guest 0155,794,32311,2025-01-18,141.85,0516,Mastercard,40,Low Confidence,Card matched;


In [23]:
no_match_result = search_transactions(
    data=transactions,
    amount_rm=999999.99,
    transaction_date="2030-01-01",
    minimum_score=40
)

print("TEST 4: NO MATCH")
print("----------------")

if no_match_result.empty:
    print("PASS: No matching transaction was returned.")
else:
    print("CHECK REQUIRED: An unexpected result was returned.")
    display(no_match_result)

TEST 4: NO MATCH
----------------
PASS: No matching transaction was returned.


In [24]:
print("TEST 5: INVALID INPUT")
print("---------------------")

try:
    invalid_input_result = search_transactions(
        data=transactions,
        card_last4="7421",
        transaction_date="not-a-date",
        minimum_score=40
    )

    print("CHECK REQUIRED: Invalid input was accepted.")

except ValueError as error:
    print("PASS: Invalid input was rejected.")
    print("Message:", error)

TEST 5: INVALID INPUT
---------------------
PASS: Invalid input was rejected.
Message: Enter the date in YYYY-MM-DD format.


### Testing Results

The matching engine successfully passed five controlled test scenarios:

| Test scenario | Expected behaviour | Result |
|---|---|---|
| Exact match | Return the correct transaction with 100 points | Passed |
| Partial match | Return the correct transaction with 70 points | Passed |
| Ambiguous match | Return multiple low-confidence candidates | Passed |
| No match | Return an empty result | Passed |
| Invalid date | Reject the input with a clear error message | Passed |

During the ambiguous-match test, the card digits `0516` appeared in three different transactions. All three records received the same low-confidence score because no amount, date, folio or room information was supplied.

This demonstrates why the last four card digits should not be treated as a unique identifier. The system presents possible candidates for staff verification instead of automatically selecting an uncertain result.

## Search Performance Benchmarking

The matching engine was benchmarked using datasets containing 100, 1,000, 10,000 and 100,000 records.

Each search was repeated 20 times to reduce the effect of temporary system fluctuations. The benchmark measures only the transaction-matching process and excludes the time taken by users to enter information or review results.

Larger benchmark datasets were created by sampling the synthetic valid transactions with replacement. Therefore, the benchmark evaluates processing speed rather than representing additional unique hotel transactions.

In [26]:
benchmark_sizes = [
    100,
    1000,
    10000,
    100000
]

number_of_runs = 20
benchmark_results = []

for dataset_size in benchmark_sizes:

    # Create a benchmark dataset of the required size
    benchmark_data = transactions.sample(
        n=dataset_size,
        replace=True,
        random_state=dataset_size
    ).reset_index(drop=True)

    # Select one record known to exist in this dataset
    benchmark_target = benchmark_data.iloc[0]

    # Warm-up search
    search_transactions(
        data=benchmark_data,
        card_last4=benchmark_target["card_last4"],
        amount_rm=benchmark_target["amount_rm"],
        transaction_date=benchmark_target["transaction_date"],
        minimum_score=40
    )

    individual_times = []

    # Repeat the search several times
    for run_number in range(number_of_runs):

        start_time = time.perf_counter()

        result = search_transactions(
            data=benchmark_data,
            card_last4=benchmark_target["card_last4"],
            amount_rm=benchmark_target["amount_rm"],
            transaction_date=benchmark_target["transaction_date"],
            minimum_score=40
        )

        elapsed_time = time.perf_counter() - start_time

        # Convert seconds to milliseconds
        individual_times.append(
            elapsed_time * 1000
        )

    benchmark_results.append({
        "dataset_size": dataset_size,
        "average_time_ms": np.mean(individual_times),
        "median_time_ms": np.median(individual_times),
        "minimum_time_ms": np.min(individual_times),
        "maximum_time_ms": np.max(individual_times),
        "successful_match": not result.empty
    })

benchmark_summary = pd.DataFrame(
    benchmark_results
)

display(
    benchmark_summary.style.format({
        "dataset_size": "{:,.0f}",
        "average_time_ms": "{:.3f}",
        "median_time_ms": "{:.3f}",
        "minimum_time_ms": "{:.3f}",
        "maximum_time_ms": "{:.3f}"
    })
)

,dataset_size,average_time_ms,median_time_ms,minimum_time_ms,maximum_time_ms,successful_match
0,100,12.338,7.297,6.762,97.651,True
1,"1,000",7.606,7.362,6.982,9.163,True
2,"10,000",8.919,8.578,7.829,13.140,True
3,"100,000",26.206,24.883,22.761,38.424,True


### Performance Findings

The matching engine successfully returned a result at every tested dataset size.

Median search time increased from 7.297 milliseconds for 100 records to 24.883 milliseconds for 100,000 records. Therefore, the engine searched 100,000 synthetic records in approximately 0.025 seconds.

The median was used as the primary performance measure because it is less affected by temporary system fluctuations. For example, one 100-record search took 97.651 milliseconds, causing its average to be higher than the average for 1,000 records.

The results indicate that the matching engine remained responsive as the dataset increased in size. However, these measurements represent processing time only. They exclude data-entry time, application loading time and the staff member’s final verification of the suggested match.

In [27]:
benchmark_file = DATA_FOLDER / "benchmark_results.csv"

benchmark_summary.to_csv(
    benchmark_file,
    index=False
)

print("Benchmark results saved to:")
print(benchmark_file)

Benchmark results saved to:
C:\Users\U S E R\Credit-card-allocation-analytics\data\benchmark_results.csv
